# 12 Impactful Event Classification: cause × persistence

Classifies **every impactful event** (the 303 IPv4 / 132 IPv6 events whose coincident peer drop clears
the AS's own 95th-percentile churn floor) along two axes, to separate the RIB-ambiguous cases from
enforcement-consistent ones (reviewer 38F/38D, key point K1):

- **cause** — are the new prefixes a *deaggregation* of space the AS already announced, *new space*, or a
  *mix*? (nb-13 containment test via `ipaddress.subnet_of`, collapsed with the 80/20 dominance rule.)
- **persistence** — does the elevated announcement *persist* or *revert*? An event persists if the announced
  count stays above the **pre-crossing baseline** for the majority of the week after the crossing. We track
  the baseline, not the limit, because an operator can raise its own limit yet keep the prefixes (BelCloud).

Only the *deaggregation ∩ reverts* cell is one where RIB-only data cannot exclude traffic engineering or
scrubbing (where BCE sits); persistent growth is consistent with genuine enforcement.
This runs on the same detailed event file as nb 11; nb 13 does the narrower critical-only root-cause figure.

In [ ]:
import os
import json
import pickle
import ipaddress
import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
REPO_ROOT = os.path.abspath("..")
with open(os.path.join(REPO_ROOT, "settings.json")) as f:
    parameters = json.load(f)
    for _k in ("DATA_DIR", "DATA_RAW_DIR", "IMAGE_DIR", "WORKING_DIR", "VISIBILITY_OUTPUT_DIR", "VISIBILITY_ANNOUNCED_OUTPUT_DIR"):
        if isinstance(parameters.get(_k), str) and not os.path.isabs(parameters[_k]):
            parameters[_k] = os.path.normpath(os.path.join(REPO_ROOT, parameters[_k]))

data_dir = parameters["DATA_DIR"]
image_dir = parameters["IMAGE_DIR"]
start_date = parameters["START_DATE"]
end_date = parameters["END_DATE"]

plt.rcParams["font.size"] = 22
ipv_color = {4: "tab:blue", 6: "tab:green"}
ipvs = [4, 6]

# cause thresholds (identical to nb 13): >=80% deagg -> Deaggregation; <=20% -> New space; else Mixed
DEAGG_DOMINANT_PCT = 80
NEW_SPACE_DOMINANT_PCT = 20

# persistence: stay above the pre-crossing baseline for >= PERSIST_FRAC of the WINDOW_DAYS after crossing
WINDOW_DAYS = 7
PERSIST_FRAC = 0.5

In [ ]:
## Open stats output file (overwrites on every run)
numbers_dir = f"{data_dir}/processed/numbers"
os.makedirs(numbers_dir, exist_ok=True)
_stats = open(f"{numbers_dir}/12-Event_classification.md", "w")
_stats.write("# Stats: 12-Event_classification\n\n")
_stats.write(f"*Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}*\n\n")
_stats.write(f"- Cause thresholds: deagg-side >= {DEAGG_DOMINANT_PCT}%, new-space-side <= {NEW_SPACE_DOMINANT_PCT}%\n")
_stats.write(f"- Persistence: above pre-crossing baseline for >= {int(PERSIST_FRAC*100)}% of the {WINDOW_DAYS} days after the crossing\n\n")
print("Stats file opened.")

## Load data

In [ ]:
# impactful events with per-event prefix lists (same file nb 11 writes)
with open(f"{data_dir}/processed/excedence_events_detailed.json") as f:
    events = {int(k): v for k, v in json.load(f).items()}

# announced-prefix time series (visibility_95) + per-date PeeringDB limits (for the baseline series)
with open(f"{data_dir}/processed/timeseries_prefix_announced_visibility.pkl", "rb") as f:
    announced_prefixes = pickle.load(f)
df_peeringdb = pd.read_pickle(
    f"{data_dir}/processed/peeringdb/prefix_limit_peeringdb_{start_date}_{end_date}.pkl"
).set_index("asn")

for ipv in ipvs:
    n = len(events[ipv])
    ncrit = sum(1 for e in events[ipv] if e.get("is_critical"))
    print(f"IPv{ipv}: {n} impactful events ({ncrit} critical), with new_prefixes populated: "
          f"{sum(1 for e in events[ipv] if e.get('new_prefixes'))}")

# the four case studies, to sanity-check their placement in the matrix
CASES = {(25273, 4): "BCE", (52920, 4): "IVOCS", (44901, 6): "BelCloud", (52603, 6): "SupplyNet"}

## Cause axis: deaggregation / new space / mixed

For each new prefix, test whether it is a subnet of any prefix the AS already announced
(`ipaddress.subnet_of`). The deagg share then collapses to a row via the 80/20 rule.

In [ ]:
def cause_row(event):
    """'Deaggregation' (>=80% of new prefixes are more-specifics of prior space),
    'New space' (<=20%), or 'Mixed' (in between). None if no parseable new prefixes."""
    new = event.get("new_prefixes", [])
    if not new:
        return None
    prev_nets = []
    for p in event.get("previous_prefixes", []):
        try:
            prev_nets.append(ipaddress.ip_network(p, strict=False))
        except ValueError:
            pass
    n_deagg = n_total = 0
    for p in new:
        try:
            net = ipaddress.ip_network(p, strict=False)
        except ValueError:
            continue
        n_total += 1
        if any(net.subnet_of(b) for b in prev_nets if b.version == net.version):
            n_deagg += 1
    if n_total == 0:
        return None
    pct = 100.0 * n_deagg / n_total
    if pct >= DEAGG_DOMINANT_PCT:
        return "Deaggregation"
    if pct <= NEW_SPACE_DOMINANT_PCT:
        return "New space"
    return "Mixed"

## Persistence axis: reverts / persists

`persist_fraction` = share of 8-hour snapshots in the week after the crossing whose announced count is
still above the pre-crossing baseline. An event **persists** if this is $\ge$ 50%, else it **reverts**.
Using the baseline (not the limit) is essential: BelCloud raised its limit but kept the prefixes.

In [ ]:
def announced_series(asn, ipv):
    if asn not in announced_prefixes or ipv not in announced_prefixes[asn]:
        return None
    return announced_prefixes[asn][ipv]["visibility_95"]


def persist_fraction(asn, ipv, cross_date, baseline):
    ann = announced_series(asn, ipv)
    if ann is None:
        return None
    cd = pd.Timestamp(cross_date)
    window = [ann[d] for d in sorted(ann)
              if cd <= pd.Timestamp(d) <= cd + pd.Timedelta(days=WINDOW_DAYS)]
    if not window:
        return None
    return float(np.mean([n > baseline for n in window]))


def run_days_above(asn, ipv, cross_date, baseline):
    """Consecutive days above baseline from the crossing until it first returns to baseline
    (a simpler duration measure, reported alongside the fraction for the distribution)."""
    ann = announced_series(asn, ipv)
    if ann is None:
        return None
    cd = pd.Timestamp(cross_date)
    post = [ann[d] for d in sorted(ann) if pd.Timestamp(d) >= cd]
    run = 0
    for n in post:
        if n > baseline:
            run += 1
        else:
            break
    return run * 8 / 24.0

## Temporal-threshold distribution

The persistence fraction is strongly bimodal: events either revert fully or stay up for the whole week, with few in between, so the 50% cut sits in a real valley rather than on a knife-edge.

In [ ]:
# per-event cause, persistence fraction, run length
records = {ipv: [] for ipv in ipvs}
for ipv in ipvs:
    for e in events[ipv]:
        ev = e["excedence_event"]
        baseline = ev["n_prefixes_previous_date"]
        row = cause_row(e)
        frac = persist_fraction(ev["asn"], ipv, ev["date"], baseline)
        rund = run_days_above(ev["asn"], ipv, ev["date"], baseline)
        if row is None or frac is None:
            continue
        records[ipv].append({"asn": ev["asn"], "cause": row, "frac": frac,
                             "run_days": rund, "critical": e.get("is_critical", False)})

# distribution figure: histogram of the persistence fraction (bimodality) + the 50% cut
plt.figure(figsize=(8, 5))
bins = np.linspace(0, 1, 21)
for ipv in ipvs:
    fr = np.array([r["frac"] for r in records[ipv]])
    plt.hist(fr, bins=bins, alpha=0.7, color=ipv_color[ipv], label=f"IPv{ipv}")
plt.axvline(PERSIST_FRAC, color="black", ls="--", lw=2)
plt.text(PERSIST_FRAC, plt.ylim()[1] * 0.9, " reverts $\\leftarrow$ | $\\rightarrow$ persists",
         ha="center", fontsize=15)
plt.xlabel("Fraction of the week above baseline")
plt.ylabel("Number of events")
plt.grid(axis="y", alpha=0.3)
plt.savefig(f"{image_dir}/new_prefixes/persistence_distribution.pdf", bbox_inches="tight", dpi=300)
plt.savefig(f"{image_dir}/new_prefixes/persistence_distribution.png", bbox_inches="tight", dpi=300)
plt.show()

# report the bimodality + duration sensitivity to the md
_stats.write("## Persistence-fraction distribution (bimodality)\n\n")
_stats.write("| IPv | n | frac<0.25 (clean revert) | 0.25-0.75 (ambiguous) | frac>=0.75 (clean persist) | persist (>=0.5) |\n")
_stats.write("|-----|---|--------------------------|-----------------------|----------------------------|-----------------|\n")
for ipv in ipvs:
    fr = np.array([r["frac"] for r in records[ipv]])
    _stats.write(f"| IPv{ipv} | {len(fr)} | {(fr<0.25).mean()*100:.0f}% | "
                 f"{((fr>=0.25)&(fr<0.75)).mean()*100:.0f}% | {(fr>=0.75).mean()*100:.0f}% | "
                 f"{(fr>=PERSIST_FRAC).mean()*100:.0f}% |\n")
_stats.write("\n## Reverts within N days (run length above baseline; threshold sensitivity)\n\n")
_stats.write("| IPv | median run (days) | reverts <=1d | <=3d | <=7d | <=14d |\n")
_stats.write("|-----|-------------------|-------------|------|------|-------|\n")
for ipv in ipvs:
    rd = np.array([r["run_days"] for r in records[ipv] if r["run_days"] is not None])
    _stats.write(f"| IPv{ipv} | {np.median(rd):.1f} | {(rd<=1).mean()*100:.0f}% | "
                 f"{(rd<=3).mean()*100:.0f}% | {(rd<=7).mean()*100:.0f}% | {(rd<=14).mean()*100:.0f}% |\n")
_stats.write("\n")
_stats.flush()
print("distribution figure + sensitivity written")

## The 3×2 matrix (cause × persistence)

In [ ]:
ROWS = ["Deaggregation", "New space", "Mixed"]
COLS = ["Reverts", "Persists"]


def build_matrix(only_critical=False):
    mat = {ipv: {r: {c: 0 for c in COLS} for r in ROWS} for ipv in ipvs}
    for ipv in ipvs:
        for r in records[ipv]:
            if only_critical and not r["critical"]:
                continue
            col = "Persists" if r["frac"] >= PERSIST_FRAC else "Reverts"
            mat[ipv][r["cause"]][col] += 1
    return mat


matrix_impactful = build_matrix(only_critical=False)

# sanity-check the four case studies
print("Case-study placement:")
for ipv in ipvs:
    for r in records[ipv]:
        if (r["asn"], ipv) in CASES:
            col = "Persists" if r["frac"] >= PERSIST_FRAC else "Reverts"
            print(f"  {CASES[(r['asn'], ipv)]:9}: {r['cause']:14} / {col}  (frac={r['frac']:.2f})")

for label, mat in [("IMPACTFUL (all)", matrix_impactful), ("CRITICAL (subset)", build_matrix(True))]:
    print(f"\n=== {label} ===")
    for ipv in ipvs:
        tot = sum(mat[ipv][r][c] for r in ROWS for c in COLS)
        print(f"IPv{ipv} (n={tot}):  {'':14}{'Reverts':>9}{'Persists':>9}")
        for r in ROWS:
            print(f"  {'':16}{r:14}{mat[ipv][r]['Reverts']:>9}{mat[ipv][r]['Persists']:>9}")

### Write the matrix to the stats file + LaTeX for the paper

In [ ]:
_stats.write("## Event Classification Matrix (cause x persistence)\n\n")
for ipv in ipvs:
    mat = matrix_impactful[ipv]
    tot = sum(mat[r][c] for r in ROWS for c in COLS)
    _stats.write(f"### IPv{ipv} (impactful, n={tot})\n\n")
    _stats.write("| Cause | Reverts | Persists |\n|-------|---------|----------|\n")
    for r in ROWS:
        _stats.write(f"| {r} | {mat[r]['Reverts']} | {mat[r]['Persists']} |\n")
    _stats.write("\n")
_stats.flush()

# LaTeX: one combined table, IPv4 and IPv6 side by side (Reverts | Persists each)
print(r"\begin{table}[t]")
print(r"\centering")
print(r"\caption{Impactful events by structural cause and persistence. Only the "
      r"deaggregation-and-reverts cell is ambiguous under RIB-only data (where BCE sits); "
      r"persistent events are consistent with genuine enforcement.}")
print(r"\label{tab:eventclass}")
print(r"\begin{tabular}{@{}lcccc@{}}")
print(r"\toprule")
print(r"& \multicolumn{2}{c}{\textbf{IPv4}} & \multicolumn{2}{c}{\textbf{IPv6}} \\")
print(r"\cmidrule(lr){2-3}\cmidrule(lr){4-5}")
print(r"\textbf{Cause} & \textbf{Reverts} & \textbf{Persists} & \textbf{Reverts} & \textbf{Persists} \\")
print(r"\midrule")
for r in ROWS:
    m4, m6 = matrix_impactful[4][r], matrix_impactful[6][r]
    print(f"{r} & {m4['Reverts']} & {m4['Persists']} & {m6['Reverts']} & {m6['Persists']} \\\\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")

In [ ]:
_stats.close()
print(f"Stats written to {numbers_dir}/12-Event_classification.md")